In [ ]:
import json
import pandas as pd

with open("master_meetings.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))

# If data is a dict
if isinstance(data, dict):

    # Common case:
    if "meetings" in data:
        df = pd.DataFrame(data["meetings"])

    else:
        df = pd.DataFrame.from_dict(
            data,
            orient="index"
        )

else:
    df = pd.DataFrame(data)

print(df.head())
print(df.columns.tolist())

print(df["overall_sentiment"].value_counts(dropna=False))
print("\n")
print(df["sentiment_score"].describe())

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------
# Load master meetings
# ------------------------

with open("master_meetings.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Your file is a dict
df = pd.DataFrame(data["meetings"]) if "meetings" in data else pd.DataFrame(data)

# ------------------------
# Load themes
# ------------------------

themes = pd.read_csv("themes_by_meeting.csv")

# ------------------------
# Theme → Call Type
# ------------------------

theme_map = {
    "customer_support_escalation": "customer_support",

    "compliance_audit": "external",
    "renewal_expansion": "external",
    "onboarding_implementation": "external",
    "product_feedback_feature_gap": "external",
    "competitive_review": "external",

    "internal_planning_roadmap": "internal"
}

themes["call_type"] = themes["assigned_theme"].map(theme_map)

# ------------------------
# Merge
# ------------------------

merged = df.merge(
    themes[["meeting_id", "call_type"]],
    on="meeting_id",
    how="inner"
)

print("Meetings:", len(merged))

# ------------------------
# Sentiment Summary
# ------------------------

summary = (
    merged.groupby("call_type")
    .agg(
        meeting_count=("meeting_id", "count"),
        avg_sentiment=("sentiment_score", "mean"),
        min_sentiment=("sentiment_score", "min"),
        max_sentiment=("sentiment_score", "max")
    )
    .reset_index()
)

print("\n===== SENTIMENT SUMMARY =====\n")
print(summary)

# ------------------------
# Sentiment Distribution
# ------------------------

distribution = (
    pd.crosstab(
        merged["call_type"],
        merged["overall_sentiment"],
        normalize="index"
    ) * 100
)

print("\n===== DISTRIBUTION (%) =====\n")
print(distribution.round(2))

# ------------------------
# Export
# ------------------------

summary.to_csv(
    "sentiment_summary.csv",
    index=False
)

distribution.to_csv(
    "sentiment_distribution.csv"
)

# ------------------------
# Plot 1
# ------------------------

summary.plot(
    x="call_type",
    y="avg_sentiment",
    kind="bar",
    legend=False
)

plt.title(
    "Average Sentiment by Call Type"
)

plt.tight_layout()

plt.savefig(
    "avg_sentiment_by_call_type.png"
)

plt.show()

# ------------------------
# Plot 2
# ------------------------

distribution.plot(
    kind="bar",
    stacked=True
)

plt.title(
    "Sentiment Distribution by Call Type"
)

plt.ylabel("Percentage")

plt.tight_layout()

plt.savefig(
    "sentiment_distribution_by_type.png"
)

plt.show()

In [ ]:
print(summary)
print(distribution)